In [79]:

import pandas as pd
from sklearn.model_selection import train_test_split

In [47]:
labels = pd.read_csv('../../data/medical_tc_labels.csv') 

In [48]:
labels.head()

,condition_label,condition_name
0,1,neoplasms
1,2,digestive system diseases
2,3,nervous system diseases
3,4,cardiovascular diseases
4,5,general pathological conditions


In [49]:
labels.shape

(5, 2)

In [50]:
labels.duplicated().sum()

np.int64(0)

In [51]:
labels.isnull().sum()

condition_label    0
condition_name     0
dtype: int64

In [52]:
train = pd.read_csv('../../data/medical_tc_train.csv') 

In [53]:
train.head()

,condition_label,medical_abstract
0,5,Tissue changes around loose prostheses. A cani...
1,1,Neuropeptide Y and neuron-specific enolase lev...
2,2,"Sexually transmitted diseases of the colon, re..."
3,1,Lipolytic factors associated with murine and h...
4,3,Does carotid restenosis predict an increased r...


In [54]:
train.duplicated().sum()

np.int64(0)

In [55]:
train.shape

(11550, 2)

In [56]:
train['condition_label'].value_counts()

condition_label
5    3844
1    2530
4    2441
3    1540
2    1195
Name: count, dtype: int64

In [57]:
train['medical_abstract'].duplicated().sum()

np.int64(2105)

In [ ]:
test = pd.read_csv('../../data/medical_tc_test.csv')

In [59]:
test.head()

,condition_label,medical_abstract
0,3,Obstructive sleep apnea following topical orop...
1,5,Neutrophil function and pyogenic infections in...
2,5,A phase II study of combined methotrexate and ...
3,1,Flow cytometric DNA analysis of parathyroid tu...
4,4,Paraneoplastic vasculitic neuropathy: a treata...


In [60]:
test.shape

(2888, 2)

In [61]:
test['medical_abstract'].duplicated().sum()

np.int64(118)

In [62]:
label_names = dict(zip(labels["condition_label"], labels["condition_name"]))


In [ ]:
URGENCY_MAP = {
    "cardiovascular diseases": "urgent",
    "nervous system diseases": "urgent",
    "neoplasms": "atention",
    "digestive system diseases": "atention",
    "general pathological conditions": "normal",
}

for df in (train, test):
    df["condition_name"] = df["condition_label"].map(label_names)
    df["urgency_label"] = df["condition_name"].map(URGENCY_MAP)

In [64]:
# marcar de qual conjunto cada linha veio, antes de juntar
train_tagged = train.copy()
train_tagged["source"] = "train"
test_tagged = test.copy()
test_tagged["source"] = "test"

combined = pd.concat([train_tagged, test_tagged], ignore_index=True)

In [65]:
# textos que aparecem em AMBOS train e test
train_texts = set(train["medical_abstract"])
test_texts = set(test["medical_abstract"])
overlap_texts = train_texts & test_texts

print(f"Textos únicos presentes em train E test: {len(overlap_texts)}")

Textos únicos presentes em train E test: 988


In [66]:
# para cada texto em overlap, checar se o urgency_label é consistente entre as ocorrências
overlap_rows = combined[combined["medical_abstract"].isin(overlap_texts)]

label_consistency = overlap_rows.groupby("medical_abstract")["urgency_label"].nunique()

same_label = (label_consistency == 1).sum()
diff_label = (label_consistency > 1).sum()

print(f"Mesmo urgency_label em todas as ocorrências: {same_label} ({same_label/len(overlap_texts)*100:.1f}%)")
print(f"Urgency_label DIFERENTE entre train e test: {diff_label} ({diff_label/len(overlap_texts)*100:.1f}%)")

Mesmo urgency_label em todas as ocorrências: 76 (7.7%)
Urgency_label DIFERENTE entre train e test: 912 (92.3%)


In [67]:
# quantas linhas do test, especificamente, sofrem do caso mais grave (label diferente)
conflicting_texts = label_consistency[label_consistency > 1].index
test_rows_conflicting = test[test["medical_abstract"].isin(conflicting_texts)]
print(f"\nLinhas do TEST com label conflitante em relação ao train: {len(test_rows_conflicting)} de {len(test)} ({len(test_rows_conflicting)/len(test)*100:.1f}%)")

# exemplo concreto, se houver
if diff_label > 0:
    example_text = conflicting_texts[0]
    print("\nExemplo de conflito real entre train e test:")
    print(combined[combined["medical_abstract"] == example_text][["source", "condition_name", "urgency_label"]])


Linhas do TEST com label conflitante em relação ao train: 934 de 2888 (32.3%)

Exemplo de conflito real entre train e test:
      source                   condition_name urgency_label
5969   train          nervous system diseases       urgente
14304   test  general pathological conditions        normal


### juntar as duas bases

In [68]:
train["source"] = "train"
test["source"] = "test"

In [69]:
pool = pd.concat([train, test], ignore_index=True)
print(f"Pool total: {len(pool)} linhas")

Pool total: 14438 linhas


In [70]:
label_consistency = pool.groupby("medical_abstract")["urgency_label"].nunique()

consistent_texts = label_consistency[label_consistency == 1].index
conflicting_texts = label_consistency[label_consistency > 1].index

print(f"Textos com label consistente: {len(consistent_texts)}")
print(f"Textos com label conflitante (serão removidos): {len(conflicting_texts)}")

Textos com label consistente: 8546
Textos com label conflitante (serão removidos): 2681


In [71]:
pool_clean = pool[pool["medical_abstract"].isin(consistent_texts)].copy()


In [72]:
pool_clean = pool_clean.drop_duplicates(subset="medical_abstract", keep="first")


In [73]:
print(f"\nDataset final limpo: {len(pool_clean)} linhas")
print("Distribuição de classes:")
print(pool_clean["urgency_label"].value_counts())
print()
print("Proporção (%):")
print((pool_clean["urgency_label"].value_counts() / len(pool_clean) * 100).round(2))


Dataset final limpo: 8546 linhas
Distribuição de classes:
urgency_label
urgente    3124
atencao    3028
normal     2394
Name: count, dtype: int64

Proporção (%):
urgency_label
urgente    36.56
atencao    35.43
normal     28.01
Name: count, dtype: float64


In [75]:
train_val, test = train_test_split(
    pool_clean,
    test_size=0.15,
    stratify=pool_clean["urgency_label"],
    random_state=42,
)

In [ ]:
val_ratio = 0.15 / 0.85
train, val = train_test_split(
    train_val,
    test_size=val_ratio,
    stratify=train_val["urgency_label"],
    random_state=42,
)

print(f"Train: {len(train)} ({len(train)/len(pool_clean)*100:.1f}%)")
print(f"Val:   {len(val)} ({len(val)/len(pool_clean)*100:.1f}%)")
print(f"Test:  {len(test)} ({len(test)/len(pool_clean)*100:.1f}%)")
print()
print("Distribuição de classes em cada conjunto:")
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"\n{name}:")
    print((df["urgency_label"].value_counts(normalize=True) * 100).round(2))

Train: 5982 (70.0%)
Val:   1282 (15.0%)
Test:  1282 (15.0%)

Distribuição de classes em cada conjunto:

Train:
urgency_label
urgente    36.54
atencao    35.44
normal     28.02
Name: proportion, dtype: float64

Val:
urgency_label
urgente    36.58
atencao    35.41
normal     28.00
Name: proportion, dtype: float64

Test:
urgency_label
urgente    36.58
atencao    35.41
normal     28.00
Name: proportion, dtype: float64


In [81]:
# selecionar só as colunas relevantes pro resto do projeto
# (mantendo 'source' para rastreabilidade, como combinado)
cols_to_save = ["medical_abstract", "urgency_label", "source"]

train[cols_to_save].to_csv("../../data/processed/train.csv", index=False)
val[cols_to_save].to_csv("../../data/processed/val.csv", index=False)
test[cols_to_save].to_csv("../../data/processed/test.csv", index=False)

print("Arquivos salvos em data/processed/:")
for name, df in [("train.csv", train), ("val.csv", val), ("test.csv", test)]:
    print(f"  {name}: {len(df)} linhas")

Arquivos salvos em data/processed/:
  train.csv: 5982 linhas
  val.csv: 1282 linhas
  test.csv: 1282 linhas


In [82]:
import re
from collections import Counter

train["n_words"] = train["medical_abstract"].str.split().str.len()

print("Comprimento (em palavras) - geral:")
print(train["n_words"].describe())
print()

print("Comprimento (em palavras) - por classe:")
print(train.groupby("urgency_label")["n_words"].describe()[["mean", "50%", "min", "max"]])


Comprimento (em palavras) - geral:
count    5982.000000
mean      180.840522
std        75.572443
min        26.000000
25%       125.000000
50%       177.000000
75%       234.000000
max       596.000000
Name: n_words, dtype: float64

Comprimento (em palavras) - por classe:
                     mean    50%   min    max
urgency_label                                
atencao        178.160849  176.0  26.0  422.0
normal         172.579952  165.0  29.0  471.0
urgente        189.772644  190.0  28.0  596.0


In [83]:
# ---- Vocabulário mais frequente por classe ----
# stopwords minimas so pra essa inspecao rapida (o TfidfVectorizer vai usar sua propria lista depois)
STOP_MINI = set(
    "the of and to in a is with for was were are on that as by an this "
    "be his her their patients".split()
)

def top_words(texts, n=20):
    words = []
    for t in texts:
        toks = re.findall(r"[a-zA-Z]+", t.lower())
        words.extend([w for w in toks if w not in STOP_MINI and len(w) > 2])
    return Counter(words).most_common(n)

for cls in ["urgente", "atencao", "normal"]:
    subset = train[train["urgency_label"] == cls]["medical_abstract"]
    print(f"\nTop 20 palavras - {cls}:")
    for word, count in top_words(subset):
        print(f"  {word}: {count}")


Top 20 palavras - urgente:
  than: 2545
  from: 1627
  less: 1536
  after: 1522
  blood: 1408
  pressure: 1386
  disease: 1299
  coronary: 1280
  not: 1274
  had: 1208
  group: 1208
  during: 1207
  these: 1104
  artery: 946
  study: 912
  ventricular: 909
  treatment: 896
  hypertension: 885
  heart: 882
  left: 877

Top 20 palavras - atencao:
  from: 1669
  than: 1649
  cancer: 1557
  cell: 1533
  cells: 1420
  tumor: 1365
  had: 1294
  not: 1255
  disease: 1232
  these: 1145
  carcinoma: 1099
  treatment: 1079
  after: 974
  tumors: 940
  have: 907
  cases: 865
  less: 855
  all: 838
  one: 831
  two: 808

Top 20 palavras - normal:
  than: 1318
  from: 1257
  after: 1097
  had: 932
  not: 899
  these: 787
  less: 755
  group: 684
  during: 679
  treatment: 642
  study: 632
  two: 620
  have: 609
  one: 606
  disease: 539
  but: 524
  may: 519
  patient: 515
  all: 512
  results: 509
